## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-21-21-50-52 +0000,startribune,Live: Pierz holds commanding lead over St. Cro...,https://www.startribune.com/live-scores-update...
1,2026-11-21-21-46-50 +0000,nyt,"When Animals Go Viral, Zoos Get Much More Than...",https://www.nytimes.com/2026/11/21/us/fiona-zo...
2,2026-11-21-21-46-35 +0000,nypost,LA City Council approves budget bailout as $32...,https://nypost.com/2026/11/21/us-news/la-city-...
3,2026-11-21-21-34-25 +0000,nypost,California prosecutor rips Newsom’s death row ...,https://nypost.com/2026/11/21/us-news/orange-c...
4,2026-11-21-21-32-09 +0000,nypost,"NYPD boss Jessica Tisch, officers gift thousan...",https://nypost.com/2026/11/21/us-news/thousand...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2719/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
47,trump,29
25,california,12
154,new,11
597,he,10
363,you,9


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
80,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...,74
190,2026-11-21-01-03-00 +0000,wsj,During President Trump’s White House briefing ...,https://www.wsj.com/politics/elections/how-a-w...,72
74,2026-11-21-16-32-51 +0000,nypost,Trump defends Stephen Miller after Hakeem Jeff...,https://nypost.com/2026/11/21/us-news/trump-de...,64
156,2026-11-21-09-53-45 +0000,nypost,FBI’s ‘Most Wanted’ cartel kingpin ‘Chapo Isid...,https://nypost.com/2026/11/21/world-news/fbis-...,63
90,2026-11-21-15-10-50 +0000,chitri,Mexico detains top cartel boss ‘El Chapo Isidr...,https://www.chicagotribune.com/2026/11/21/el-c...,62


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
80,74,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...
156,54,2026-11-21-09-53-45 +0000,nypost,FBI’s ‘Most Wanted’ cartel kingpin ‘Chapo Isid...,https://nypost.com/2026/11/21/world-news/fbis-...
190,43,2026-11-21-01-03-00 +0000,wsj,During President Trump’s White House briefing ...,https://www.wsj.com/politics/elections/how-a-w...
49,42,2026-11-21-19-08-58 +0000,nypost,SpaceX shuttle to splash down off California c...,https://nypost.com/2026/11/21/us-news/spacex-d...
221,37,2026-11-20-23-08-57 +0000,bbc,"In a year of AI solving maths problems, an Ant...",https://www.bbc.co.uk/news/articles/c6pqr2g7w0...
60,36,2026-11-21-18-19-55 +0000,chitri,The Biz Week That Was: Data center gets tax br...,https://www.chicagotribune.com/2026/11/21/the-...
74,35,2026-11-21-16-32-51 +0000,nypost,Trump defends Stephen Miller after Hakeem Jeff...,https://nypost.com/2026/11/21/us-news/trump-de...
65,35,2026-11-21-17-38-58 +0000,nypost,Russian cargo ship bursts into flames in Balti...,https://nypost.com/2026/11/21/world-news/russi...
51,29,2026-11-21-19-00-34 +0000,nypost,Mayor of major California city issues dire wea...,https://nypost.com/2026/11/21/us-news/sf-mayor...
206,29,2026-11-21-00-25-49 +0000,cbc,U.S. senator says Canada should make interim t...,https://www.cbc.ca/news/politics/us-canada-int...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
